# Visualization — Backup Experiments (Pre-LoRA / Previous Runs)

Loads `all_results.pkl` and `all_preds.pkl` from the three backup folders and generates:

1. **5×5 cross-database matrices** — ROC-AUC, Accuracy, F1-FAKE  
2. **ROC curves** — per training database  
3. **Confusion matrices** — 5×5 grid  
4. **Per-class metrics bar chart** — Precision/Recall/F1 per training target  
5. **Side-by-side model comparison** — best diagonal (within-DB) score per model  

All figures are saved to the backup folder and synced to Google Drive.

| Model | Backup path | Evaluations |
|---|---|---|
| CNN-SRM | `backups/cnn_srm_previous_run/` | 25 / 25 ✓ |
| EfficientNet | `backups/efficientnet_previous_run/` | 10 / 25 (partial) |
| ViT-Tiny | `backups/vit_previous_run/` | 25 / 25 ✓ |

In [ ]:
# =================== CELL 1: SETUP ===================
import os
import pickle
import numpy as np
import matplotlib
matplotlib.use('Agg')   # headless — saves PNGs even without display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pandas as pd
import subprocess
from sklearn.metrics import roc_curve, confusion_matrix

# ── Backup paths ──────────────────────────────────────────────────────────
BACKUP_ROOT = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/backups'

MODELS = {
    'CNN-SRM': {
        'folder':       os.path.join(BACKUP_ROOT, 'cnn_srm_previous_run'),
        'results_file': 'all_results.pkl',
        'preds_file':   'all_preds.pkl',
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
        'color':        '#e6194b',
    },
    'EfficientNet': {
        'folder':       os.path.join(BACKUP_ROOT, 'efficientnet_previous_run'),
        'results_file': 'all_results.pkl',
        'preds_file':   'all_preds.pkl',
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
        'color':        '#3cb44b',
    },
    'ViT-Tiny': {
        'folder':       os.path.join(BACKUP_ROOT, 'vit_previous_run'),
        'results_file': 'all_results.pkl',
        'preds_file':   'all_predictions.pkl',   # ViT uses a different filename
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'Combined'],
        'color':        '#4363d8',
    },
}

# ── Load all data ─────────────────────────────────────────────────────────
DATA = {}
for model_name, cfg in MODELS.items():
    r_path = os.path.join(cfg['folder'], cfg['results_file'])
    p_path = os.path.join(cfg['folder'], cfg['preds_file'])
    results = pickle.load(open(r_path, 'rb')) if os.path.exists(r_path) else {}
    preds   = pickle.load(open(p_path, 'rb')) if os.path.exists(p_path) else {}
    total   = sum(len(v) for v in results.values())
    DATA[model_name] = {'results': results, 'preds': preds, 'total': total}
    print(f'{model_name}: {total} evaluations loaded from {cfg["folder"]}')

print('\nSetup complete.')

In [ ]:
# =================== CELL 2: CROSS-DATABASE RESULTS MATRICES ===================
# For each model that has results, plot ROC-AUC / Accuracy / F1-FAKE heatmaps.

METRICS_TO_PLOT = [
    ('roc_auc',  'ROC-AUC',  'YlOrRd'),
    ('accuracy', 'Accuracy', 'YlGn'),
    ('f1_fake',  'F1-FAKE',  'Blues'),
]

for model_name, cfg in MODELS.items():
    results = DATA[model_name]['results']
    dbs     = cfg['dbs']
    folder  = cfg['folder']

    if not results:
        print(f'[SKIP] {model_name}: no results available')
        continue

    available_rows = [d for d in dbs if d in results]
    print(f'\n{model_name} — {len(available_rows)}/{len(dbs)} training rows available')

    for metric_key, metric_label, cmap in METRICS_TO_PLOT:
        matrix = [
            [results[tr].get(te, {}).get(metric_key, float('nan'))
             for te in dbs]
            for tr in available_rows
        ]
        df = pd.DataFrame(
            matrix,
            index=[f'Train: {d}' for d in available_rows],
            columns=[f'Test: {d}' for d in dbs],
        )

        fig, ax = plt.subplots(figsize=(max(9, len(dbs) * 2), max(5, len(available_rows) * 1.5)))
        sns.heatmap(
            df.astype(float), annot=True, fmt='.3f',
            cmap=cmap, vmin=0.40, vmax=1.00, ax=ax,
            linewidths=0.5, linecolor='gray',
            cbar_kws={'label': metric_label}
        )
        ax.set_title(
            f'{model_name} — Cross-Database {metric_label} (Backup / Previous Run)\n'
            'Diagonal = within-DB · Off-diagonal = cross-DB generalization',
            fontsize=12, fontweight='bold'
        )
        ax.set_ylabel('Training Database', fontsize=11)
        ax.set_xlabel('Test Database', fontsize=11)
        plt.xticks(rotation=30, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()

        model_slug = model_name.lower().replace('-', '_').replace(' ', '_')
        fig_path = os.path.join(folder, f'cross_db_{metric_key}.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Saved: {fig_path}')

        print(f'\n{metric_label} Matrix ({model_name}):')
        print(df.to_string())
        print()

In [ ]:
# =================== CELL 3: ROC CURVES (per model, per training DB) ===================

DB_COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4']

for model_name, cfg in MODELS.items():
    results = DATA[model_name]['results']
    preds   = DATA[model_name]['preds']
    dbs     = cfg['dbs']
    folder  = cfg['folder']

    if not preds:
        print(f'[SKIP] {model_name}: no predictions available')
        continue

    color_map = {db: c for db, c in zip(dbs, DB_COLORS)}
    available_train = [d for d in dbs if d in preds]

    for train_db in available_train:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random (AUC=0.50)')

        for test_db in dbs:
            if test_db not in preds.get(train_db, {}):
                continue
            y_true = np.array(preds[train_db][test_db]['y_true'])
            y_prob = np.array(preds[train_db][test_db]['y_prob'])
            fpr, tpr, _ = roc_curve(y_true, y_prob)
            auc_val   = results.get(train_db, {}).get(test_db, {}).get('roc_auc', 0)
            is_within = (train_db == test_db)
            label     = f'{test_db}  (AUC={auc_val:.3f})' + ('  ← within-DB' if is_within else '')
            ax.plot(fpr, tpr,
                    color=color_map.get(test_db, 'gray'),
                    lw=3.0 if is_within else 1.5,
                    ls='-'  if is_within else '--',
                    label=label)

        ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.02])
        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate',  fontsize=12)
        ax.set_title(f'{model_name} — ROC Curves (Backup)\nTrained on: {train_db}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='lower right', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()

        slug = train_db.lower().replace(' ', '_')
        fig_path = os.path.join(folder, f'roc_{slug}.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  Saved ROC: {fig_path}')

In [ ]:
# =================== CELL 4: CONFUSION MATRICES (all train×test pairs per model) ===================

for model_name, cfg in MODELS.items():
    results = DATA[model_name]['results']
    preds   = DATA[model_name]['preds']
    dbs     = cfg['dbs']
    folder  = cfg['folder']

    if not preds:
        print(f'[SKIP] {model_name}: no predictions')
        continue

    available_train = [d for d in dbs if d in preds]
    n_train = len(available_train)
    n_test  = len(dbs)

    fig, axes = plt.subplots(n_train, n_test,
                              figsize=(4 * n_test, 3.5 * n_train))
    if n_train == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(
        f'{model_name} — Confusion Matrices (Backup)\n'
        'Rows = Train DB · Cols = Test DB',
        fontsize=15, fontweight='bold', y=1.01
    )

    for r, train_db in enumerate(available_train):
        for c, test_db in enumerate(dbs):
            ax = axes[r][c]
            if test_db not in preds.get(train_db, {}):
                ax.axis('off')
                ax.set_title(f'Tr:{train_db}\nTe:{test_db}', fontsize=7)
                continue

            threshold = results.get(train_db, {}).get(test_db, {}).get('threshold', 0.5)
            y_true = np.array(preds[train_db][test_db]['y_true'])
            y_pred = (np.array(preds[train_db][test_db]['y_prob']) >= threshold).astype(int)
            cm      = confusion_matrix(y_true, y_pred)
            cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

            ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
            ax.set_xticks([0, 1]); ax.set_xticklabels(['REAL', 'FAKE'], fontsize=7)
            ax.set_yticks([0, 1]); ax.set_yticklabels(['REAL', 'FAKE'], fontsize=7)
            for i in range(2):
                for j in range(2):
                    color = 'white' if cm_norm[i, j] > 0.6 else 'black'
                    ax.text(j, i, str(cm[i, j]),
                            ha='center', va='center',
                            fontsize=9, fontweight='bold', color=color)

            is_within = (train_db == test_db)
            title_color = '#c00000' if is_within else 'black'
            acc = results.get(train_db, {}).get(test_db, {}).get('accuracy', float('nan'))
            ax.set_title(f'Tr:{train_db}\nTe:{test_db}\nAcc={acc:.3f}',
                         fontsize=7, color=title_color,
                         fontweight='bold' if is_within else 'normal')

    plt.tight_layout()
    fig_path = os.path.join(folder, 'confusion_matrices.png')
    plt.savefig(fig_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'{model_name}: Saved → {fig_path}')

In [ ]:
# =================== CELL 5: PER-CLASS METRICS BAR CHART ===================

METRICS_BAR = [
    ('precision_fake', 'Prec-FAKE',  '#e6194b'),
    ('recall_fake',    'Rec-FAKE',   '#f58231'),
    ('f1_fake',        'F1-FAKE',    '#3cb44b'),
    ('f1_real',        'F1-REAL',    '#4363d8'),
    ('roc_auc',        'ROC-AUC',    '#911eb4'),
    ('accuracy',       'Accuracy',   '#42d4f4'),
]

for model_name, cfg in MODELS.items():
    results = DATA[model_name]['results']
    dbs     = cfg['dbs']
    folder  = cfg['folder']

    if not results:
        continue

    available_train = [d for d in dbs if d in results]

    for train_db in available_train:
        test_dbs  = [t for t in dbs if t in results.get(train_db, {})]
        n_tests   = len(test_dbs)
        n_metrics = len(METRICS_BAR)
        x         = np.arange(n_tests)
        bar_w     = 0.13
        offsets   = np.linspace(-(n_metrics-1)/2, (n_metrics-1)/2, n_metrics) * bar_w

        fig, ax = plt.subplots(figsize=(max(10, n_tests * 2), 6))

        for i, (key, label, color) in enumerate(METRICS_BAR):
            values = [results[train_db][t].get(key, float('nan')) for t in test_dbs]
            bars   = ax.bar(x + offsets[i], values, bar_w,
                            label=label, color=color, alpha=0.85, edgecolor='white')
            for bar, val in zip(bars, values):
                if not np.isnan(val):
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            bar.get_height() + 0.01,
                            f'{val:.2f}', ha='center', va='bottom',
                            fontsize=6, rotation=90)

        # Highlight within-DB group
        if train_db in test_dbs:
            within_idx = test_dbs.index(train_db)
            ax.axvspan(within_idx - 0.4, within_idx + 0.4,
                       alpha=0.08, color='gold', label='Within-DB')

        ax.set_xticks(x); ax.set_xticklabels(test_dbs, fontsize=10)
        ax.set_ylim(0, 1.25)
        ax.set_ylabel('Score', fontsize=11)
        ax.set_xlabel('Test Database', fontsize=11)
        ax.set_title(f'{model_name} — Per-Class Metrics (Backup)\nTrained on: {train_db}',
                     fontsize=13, fontweight='bold')
        ax.legend(loc='upper right', fontsize=8, ncol=3)
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()

        slug = train_db.lower().replace(' ', '_')
        fig_path = os.path.join(folder, f'metrics_bar_{slug}.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'  {model_name}/{train_db}: saved → {fig_path}')

In [ ]:
# =================== CELL 6: PREDICTION SCORE HISTOGRAMS (5×5 per model) ===================
# Blue bars = REAL score distribution, Orange = FAKE

COLORS_HIST = {'REAL': '#4C72B0', 'FAKE': '#DD8452'}

for model_name, cfg in MODELS.items():
    preds  = DATA[model_name]['preds']
    dbs    = cfg['dbs']
    folder = cfg['folder']

    if not preds:
        continue

    available_train = [d for d in dbs if d in preds]
    n_rows = len(available_train)
    n_cols = len(dbs)

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(4 * n_cols, 3 * n_rows), squeeze=False)
    fig.suptitle(
        f'{model_name} — Prediction Score Distributions (Backup)\n'
        'Blue = REAL (ideal → 0)   Orange = FAKE (ideal → 1)',
        fontsize=13, fontweight='bold', y=1.01
    )

    for r, train_db in enumerate(available_train):
        for c, test_db in enumerate(dbs):
            ax = axes[r][c]
            entry = preds.get(train_db, {}).get(test_db)
            if entry is None:
                ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                        transform=ax.transAxes, color='gray')
                ax.set_xticks([]); ax.set_yticks([])
            else:
                y_true = np.array(entry['y_true'])
                y_prob = np.array(entry['y_prob'])
                real_scores = y_prob[y_true == 0]
                fake_scores = y_prob[y_true == 1]
                bins = np.linspace(0, 1, 25)
                ax.hist(real_scores, bins=bins, alpha=0.6,
                        color=COLORS_HIST['REAL'], density=True)
                ax.hist(fake_scores, bins=bins, alpha=0.6,
                        color=COLORS_HIST['FAKE'], density=True)
                ax.set_xlim(0, 1)
                ax.set_xticks([0, 0.5, 1])
                is_within = (train_db == test_db)
                ax.set_facecolor('#fff8e0' if is_within else 'white')
            if c == 0:
                ax.set_ylabel(f'Tr:{train_db}', fontsize=7, fontweight='bold')
            if r == 0:
                ax.set_title(f'Te:{test_db}', fontsize=7, fontweight='bold')

    real_patch = mpatches.Patch(color=COLORS_HIST['REAL'], alpha=0.6, label='REAL')
    fake_patch = mpatches.Patch(color=COLORS_HIST['FAKE'], alpha=0.6, label='FAKE')
    fig.legend(handles=[real_patch, fake_patch], loc='upper right',
               bbox_to_anchor=(1.0, 1.0), fontsize=9)
    plt.tight_layout()

    fig_path = os.path.join(folder, 'score_histograms.png')
    plt.savefig(fig_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'{model_name}: saved histograms → {fig_path}')

In [ ]:
# =================== CELL 7: CROSS-MODEL COMPARISON (diagonal / within-DB scores) ===================
# Shows how each model performs on its own training database (diagonal)
# and overall average ROC-AUC across all test databases.

summary_rows = []

for model_name, cfg in MODELS.items():
    results = DATA[model_name]['results']
    dbs     = cfg['dbs']
    if not results:
        continue

    for train_db in dbs:
        if train_db not in results:
            continue
        within_auc  = results[train_db].get(train_db, {}).get('roc_auc',  float('nan'))
        within_f1   = results[train_db].get(train_db, {}).get('f1_fake',  float('nan'))
        within_acc  = results[train_db].get(train_db, {}).get('accuracy', float('nan'))
        all_aucs    = [v.get('roc_auc', float('nan'))
                       for v in results[train_db].values()]
        avg_auc     = float(np.nanmean(all_aucs))
        summary_rows.append({
            'Model':       model_name,
            'Train DB':    train_db,
            'Within-AUC':  within_auc,
            'Within-F1':   within_f1,
            'Within-Acc':  within_acc,
            'Avg AUC (all test DBs)': avg_auc,
        })

df_summary = pd.DataFrame(summary_rows)
print('\n=== Cross-Model Summary (Backup Experiments) ===')
print(df_summary.to_string(index=False, float_format='{:.3f}'.format))

# ── Bar chart: avg ROC-AUC per model (averaged across all train DBs) ──────
model_avg = df_summary.groupby('Model')['Avg AUC (all test DBs)'].mean().reset_index()
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_avg['Model'], model_avg['Avg AUC (all test DBs)'],
               color=[MODELS[m]['color'] for m in model_avg['Model']],
               width=0.5, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, model_avg['Avg AUC (all test DBs)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Average ROC-AUC (all test DBs)', fontsize=12)
ax.set_title('Cross-Model Comparison (Backup Experiments)\nAverage ROC-AUC across all training targets',
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(BACKUP_ROOT, 'model_comparison_avg_auc.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\nComparison chart saved → {fig_path}')

# Save summary CSV
csv_path = os.path.join(BACKUP_ROOT, 'backup_experiments_summary.csv')
df_summary.to_csv(csv_path, index=False)
print(f'Summary CSV saved → {csv_path}')

In [ ]:
# =================== CELL 8: SYNC TO GOOGLE DRIVE ===================
# Syncs all generated PNG figures and the CSV summary to Drive.

import subprocess

gdrive_dest = 'gdrive:deepfake_image_project/models/backups'
print(f'Syncing {BACKUP_ROOT}')
print(f'     to {gdrive_dest} ...')

result = subprocess.run(
    ['rclone', 'sync', BACKUP_ROOT, gdrive_dest, '--progress', '--stats-one-line'],
    capture_output=False, text=True
)
if result.returncode == 0:
    print('\nGoogle Drive sync complete.')
    print('All figures and CSV are now on Drive under backups/')
else:
    print(f'\nWARNING: rclone sync failed (exit {result.returncode}).')
    print(f'Files are safe locally at {BACKUP_ROOT}')

---

## Comparison: Backup (Previous) vs Current (LoRA) Experiments

The cells below load the **current LoRA results** and produce side-by-side comparisons for every cross-database pair where both runs have data.

**Color convention for delta charts:**
- Green = LoRA improved over backup
- Red   = LoRA regressed vs backup
- Grey  = data missing in one of the runs

In [ ]:
# =================== CELL 9: LOAD CURRENT (LoRA) RESULTS ===================

CURRENT_PATHS = {
    'CNN-SRM': {
        'folder':       '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_experiment/cnn_srm/',
        'results_file': 'all_results.pkl',
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
    },
    'EfficientNet': {
        'folder':       '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_experiment/efficientnet/',
        'results_file': 'all_results.pkl',
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'ALL'],
    },
    'ViT-Tiny': {
        'folder':       '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/RealEyes_experiment/vit/',
        'results_file': 'all_results.pkl',
        'dbs':          ['OpenForensics', 'CustomWar', 'CelebDF', 'CiFake', 'Combined'],
    },
}

CURRENT_DATA = {}
for model_name, cfg in CURRENT_PATHS.items():
    r_path  = os.path.join(cfg['folder'], cfg['results_file'])
    results = pickle.load(open(r_path, 'rb')) if os.path.exists(r_path) else {}
    total   = sum(len(v) for v in results.values())
    CURRENT_DATA[model_name] = {'results': results, 'dbs': cfg['dbs']}
    print(f'{model_name} (current/LoRA): {total} evaluations loaded')

COMPARE_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/comparison_backup_vs_lora'
os.makedirs(COMPARE_DIR, exist_ok=True)
print(f'\nFigures saved to: {COMPARE_DIR}')


In [ ]:
# =================== CELL 10: DELTA HEATMAPS (LoRA - Backup) ===================
# Green = LoRA improved, Red = LoRA regressed

DELTA_METRICS = [
    ('roc_auc',  'Delta ROC-AUC  (LoRA - Backup)',  'RdYlGn'),
    ('f1_fake',  'Delta F1-FAKE  (LoRA - Backup)',   'RdYlGn'),
    ('accuracy', 'Delta Accuracy (LoRA - Backup)',   'RdYlGn'),
]

for model_name in MODELS:
    backup_results  = DATA[model_name]['results']
    current_results = CURRENT_DATA[model_name]['results']
    dbs             = MODELS[model_name]['dbs']
    folder          = MODELS[model_name]['folder']

    available_train = [d for d in dbs if d in backup_results and d in current_results]
    if not available_train:
        print(f'[SKIP] {model_name}: no overlapping train DBs')
        continue

    for metric_key, title_suffix, cmap in DELTA_METRICS:
        matrix = []
        for tr in available_train:
            row = []
            for te in dbs:
                cur = current_results.get(tr, {}).get(te, {}).get(metric_key)
                bck = backup_results.get(tr, {}).get(te, {}).get(metric_key)
                row.append(cur - bck if (cur is not None and bck is not None) else float('nan'))
            matrix.append(row)

        df_d = pd.DataFrame(matrix,
                             index=[f'Train: {d}' for d in available_train],
                             columns=[f'Test: {d}' for d in dbs])
        fig, ax = plt.subplots(figsize=(max(9, len(dbs)*2), max(5, len(available_train)*1.5)))
        sns.heatmap(df_d.astype(float), annot=True, fmt='+.3f',
                    cmap=cmap, center=0, vmin=-0.15, vmax=0.15, ax=ax,
                    linewidths=0.5, linecolor='gray',
                    cbar_kws={'label': title_suffix})
        ax.set_title(f'{model_name} — {title_suffix}\n'
                     'Green=LoRA better · Red=LoRA worse · Grey=data missing',
                     fontsize=12, fontweight='bold')
        ax.set_ylabel('Training Database', fontsize=11)
        ax.set_xlabel('Test Database', fontsize=11)
        plt.xticks(rotation=30, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        model_slug = model_name.lower().replace('-','_').replace(' ','_')
        fig_path = os.path.join(COMPARE_DIR, f'delta_{model_slug}_{metric_key}.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fig_path}')


In [ ]:
# =================== CELL 11: SIDE-BY-SIDE ABSOLUTE HEATMAPS ===================
# Left = Backup, Right = LoRA. Metric: ROC-AUC and F1-FAKE.

for model_name in MODELS:
    backup_results  = DATA[model_name]['results']
    current_results = CURRENT_DATA[model_name]['results']
    dbs             = MODELS[model_name]['dbs']

    available_train = [d for d in dbs if d in backup_results or d in current_results]
    if not available_train:
        continue

    for metric_key, metric_label in [('roc_auc', 'ROC-AUC'), ('f1_fake', 'F1-FAKE')]:
        fig, axes = plt.subplots(1, 2,
                                  figsize=(max(18, len(dbs)*4), max(5, len(available_train)*1.4)),
                                  sharey=True)
        for ax, rsrc, run_label in [
            (axes[0], backup_results,  'Backup (Previous)'),
            (axes[1], current_results, 'Current (LoRA)'),
        ]:
            mat = [[rsrc.get(tr,{}).get(te,{}).get(metric_key, float('nan'))
                    for te in dbs] for tr in available_train]
            df  = pd.DataFrame(mat,
                                index=[f'Train: {d}' for d in available_train],
                                columns=[f'Test: {d}' for d in dbs])
            mask = df.isna()
            sns.heatmap(df.astype(float), annot=True, fmt='.3f',
                        cmap='YlOrRd', vmin=0.40, vmax=1.00, ax=ax,
                        linewidths=0.5, linecolor='gray', mask=mask,
                        cbar_kws={'label': metric_label})
            for ri in range(len(available_train)):
                for ci in range(len(dbs)):
                    if mask.iloc[ri, ci]:
                        ax.add_patch(plt.Rectangle((ci, ri), 1, 1,
                                                    fill=True, color='#cccccc', lw=0))
            ax.set_title(f'{run_label}\n{metric_label}', fontsize=12, fontweight='bold')
            ax.set_xlabel('Test Database', fontsize=10)
            ax.tick_params(axis='x', rotation=30)
            ax.tick_params(axis='y', rotation=0)
        axes[0].set_ylabel('Training Database', fontsize=11)
        fig.suptitle(f'{model_name} — {metric_label}: Backup vs LoRA',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        model_slug = model_name.lower().replace('-','_').replace(' ','_')
        fig_path = os.path.join(COMPARE_DIR, f'sidebyside_{model_slug}_{metric_key}.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: {fig_path}')


In [ ]:
# =================== CELL 12: WITHIN-DB (DIAGONAL) COMPARISON ===================
# Bar chart: pairs [Backup, LoRA] for each (model, train_db) — within-DB only.

MODEL_COLOR = {'CNN-SRM': '#e6194b', 'EfficientNet': '#3cb44b', 'ViT-Tiny': '#4363d8'}

for metric_key, metric_label in [('roc_auc', 'ROC-AUC'), ('f1_fake', 'F1-FAKE'), ('accuracy', 'Accuracy')]:
    fig, ax = plt.subplots(figsize=(14, 6))
    all_labels, backup_vals, lora_vals, col_list = [], [], [], []

    for model_name in MODELS:
        dbs  = MODELS[model_name]['dbs']
        col  = MODEL_COLOR[model_name]
        for db in dbs:
            bck = DATA[model_name]['results'].get(db, {}).get(db, {}).get(metric_key, float('nan'))
            cur = CURRENT_DATA[model_name]['results'].get(db, {}).get(db, {}).get(metric_key, float('nan'))
            all_labels.append(f'{model_name}\n{db}')
            backup_vals.append(bck)
            lora_vals.append(cur)
            col_list.append(col)

    x  = np.arange(len(all_labels))
    bw = 0.38
    ax.bar(x - bw/2, backup_vals, bw, label='Backup', color=col_list, alpha=0.5, hatch='//', edgecolor='white')
    bars_l = ax.bar(x + bw/2, lora_vals, bw, label='LoRA (Current)', color=col_list, alpha=0.9, edgecolor='white')

    for i, (cur, bck) in enumerate(zip(lora_vals, backup_vals)):
        if not np.isnan(cur):
            delta = cur - bck if not np.isnan(bck) else float('nan')
            ds = f'{delta:+.3f}' if not np.isnan(delta) else ''
            fc = '#006400' if (not np.isnan(delta) and delta > 0) else ('#8b0000' if (not np.isnan(delta) and delta < 0) else 'black')
            ax.text(x[i]+bw/2, cur+0.005, f'{cur:.3f}\n{ds}', ha='center', va='bottom',
                    fontsize=6.5, color=fc, fontweight='bold')

    sep = -0.5
    for mn in MODELS:
        sz = len(MODELS[mn]['dbs'])
        ax.axvline(sep+sz, color='gray', lw=1.2, ls='--', alpha=0.5)
        ax.text(sep+sz/2, 1.22, mn, ha='center', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc=MODEL_COLOR[mn], alpha=0.15))
        sep += sz

    ax.set_xticks(x); ax.set_xticklabels(all_labels, fontsize=7.5)
    ax.set_ylim(0, 1.30)
    ax.set_ylabel(metric_label, fontsize=12)
    ax.set_title(f'Within-DB {metric_label}: Backup vs LoRA\n'
                 '(Model tested on its own training DB — diagonal of cross-DB matrix)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig_path = os.path.join(COMPARE_DIR, f'diagonal_comparison_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')


In [ ]:
# =================== CELL 13: CROSS-DB GENERALISATION COMPARISON ===================
# Average off-diagonal metric: how models trained on DB_A perform on DB_B (B!=A).

for metric_key, metric_label in [('roc_auc', 'ROC-AUC'), ('f1_fake', 'F1-FAKE')]:
    rows = []
    for model_name in MODELS:
        dbs = MODELS[model_name]['dbs']
        for tr in dbs:
            bck_off = [DATA[model_name]['results'].get(tr,{}).get(te,{}).get(metric_key, float('nan'))
                       for te in dbs if te != tr]
            cur_off = [CURRENT_DATA[model_name]['results'].get(tr,{}).get(te,{}).get(metric_key, float('nan'))
                       for te in dbs if te != tr]
            rows.append({'Model': model_name, 'Train DB': tr,
                          f'Backup {metric_label}': float(np.nanmean(bck_off)),
                          f'LoRA {metric_label}'  : float(np.nanmean(cur_off))})

    df_c = pd.DataFrame(rows)
    df_c['Delta'] = df_c[f'LoRA {metric_label}'] - df_c[f'Backup {metric_label}']
    print(f'\n=== Off-diagonal {metric_label} ===')
    print(df_c.to_string(index=False, float_format='{:.3f}'.format))

    fig, ax = plt.subplots(figsize=(14, 6))
    labels = [f'{r.Model}\n{r["Train DB"]}' for _, r in df_c.iterrows()]
    x  = np.arange(len(labels))
    bw = 0.38
    ax.bar(x-bw/2, df_c[f'Backup {metric_label}'], bw, label='Backup', alpha=0.55, hatch='//', color='steelblue', edgecolor='white')
    ax.bar(x+bw/2, df_c[f'LoRA {metric_label}'],   bw, label='LoRA',   alpha=0.90, color='steelblue', edgecolor='white')
    for i, row in df_c.iterrows():
        d = row['Delta']
        if not np.isnan(d):
            c = '#006400' if d > 0 else '#8b0000'
            yval = row[f'LoRA {metric_label}']
            ax.text(i+bw/2, (yval if not np.isnan(yval) else 0)+0.005,
                    f'{d:+.3f}', ha='center', va='bottom', fontsize=7, color=c, fontweight='bold')

    sep3 = -0.5
    for mn in MODELS:
        sz = len(MODELS[mn]['dbs'])
        ax.axvline(sep3+sz, color='gray', lw=1.2, ls='--', alpha=0.5)
        ax.text(sep3+sz/2, 1.05, mn, ha='center', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='lightgray', alpha=0.6))
        sep3 += sz
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=7.5)
    ax.set_ylim(0, 1.20)
    ax.set_ylabel(f'Avg {metric_label} (off-diagonal)', fontsize=12)
    ax.set_title(f'Cross-DB Generalisation — {metric_label}: Backup vs LoRA\n'
                 'Higher = better transfer to unseen databases',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    fig_path = os.path.join(COMPARE_DIR, f'crossdb_generalisation_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')


In [ ]:
# =================== CELL 14: SUMMARY TABLE & CONCLUSIONS ===================
import warnings; warnings.filterwarnings('ignore')

METRICS_SUM = [('roc_auc','ROC-AUC'),('f1_fake','F1-FAKE'),('accuracy','Accuracy'),
               ('precision_fake','Prec-FAKE'),('recall_fake','Rec-FAKE')]

summary_rows = []
for model_name in MODELS:
    bres = DATA[model_name]['results']
    cres = CURRENT_DATA[model_name]['results']
    dbs  = MODELS[model_name]['dbs']
    paired = {mk: {'b': [], 'l': []} for mk, _ in METRICS_SUM}
    for tr in dbs:
        for te in dbs:
            br = bres.get(tr,{}).get(te)
            cr = cres.get(tr,{}).get(te)
            if br and cr:
                for mk, _ in METRICS_SUM:
                    b = br.get(mk); c = cr.get(mk)
                    if b is not None and c is not None:
                        paired[mk]['b'].append(b)
                        paired[mk]['l'].append(c)
    row = {'Model': model_name, 'Paired evals': len(paired['roc_auc']['b'])}
    for mk, ml in METRICS_SUM:
        if paired[mk]['b']:
            ba = float(np.mean(paired[mk]['b']))
            la = float(np.mean(paired[mk]['l']))
            row[f'Bck {ml}'] = ba; row[f'LoRA {ml}'] = la; row[f'D {ml}'] = la - ba
        else:
            row[f'Bck {ml}'] = float('nan'); row[f'LoRA {ml}'] = float('nan'); row[f'D {ml}'] = float('nan')
    summary_rows.append(row)

df_final = pd.DataFrame(summary_rows)
print('='*70)
print('SUMMARY: avg across ALL paired (train_db, test_db) combinations')
print('='*70)
for _, row in df_final.iterrows():
    n = int(row['Paired evals'])
    print(f'\n  {row["Model"]}  ({n} paired pairs)')
    for mk, ml in METRICS_SUM:
        b = row[f'Bck {ml}']; l = row[f'LoRA {ml}']; d = row[f'D {ml}']
        if not np.isnan(b):
            arr = "+" if d > 0 else ("-" if d < 0 else "=")
            print(f'    {ml:12s}: Backup={b:.4f}  LoRA={l:.4f}  Delta={d:+.4f} {arr}')

# Visual delta table
delta_cols = [f'D {ml}' for _, ml in METRICS_SUM]
df_dlt = df_final[['Model']+delta_cols].set_index('Model')
fig, ax = plt.subplots(figsize=(13, 3.5))
import matplotlib
cmap_t  = plt.cm.RdYlGn
norm_t  = matplotlib.colors.Normalize(vmin=-0.08, vmax=0.08)
ctext   = [[f'{v:+.4f}' if not np.isnan(v) else 'N/A' for v in row] for row in df_dlt.values]
ccolors = [[matplotlib.colors.to_hex(cmap_t(norm_t(v))) if not np.isnan(v) else '#dddddd'
            for v in row] for row in df_dlt.values]
tbl = ax.table(cellText=ctext, rowLabels=df_dlt.index.tolist(),
               colLabels=[c.replace('D ','Delta\n') for c in delta_cols],
               cellColours=ccolors, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.3, 2.0)
ax.axis('off')
ax.set_title('Delta = LoRA - Backup  (green=improvement, red=regression)',
             fontsize=12, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = os.path.join(COMPARE_DIR, 'summary_delta_table.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
csv_path = os.path.join(COMPARE_DIR, 'comparison_summary.csv')
df_final.to_csv(csv_path, index=False)
print(f'\nSummary CSV: {csv_path}')
print(f'Delta table: {fig_path}')


In [ ]:
# =================== CELL 15: SYNC TO GOOGLE DRIVE ===================
import subprocess

GDRIVE_COMPARE = 'gdrive:deepfake_image_project/models/comparison_backup_vs_lora'
print(f'Syncing {COMPARE_DIR}')
print(f'     to {GDRIVE_COMPARE} ...')

result = subprocess.run(
    ['rclone', 'sync', COMPARE_DIR, GDRIVE_COMPARE, '--progress', '--stats-one-line'],
    capture_output=False, text=True
)
if result.returncode == 0:
    print('\nGoogle Drive sync complete.')
    print(f'All comparison figures and CSV on Drive under:\n  {GDRIVE_COMPARE}')
else:
    print(f'\nWARNING: rclone failed (exit {result.returncode}). Files safe locally at {COMPARE_DIR}')
